In [ ]:

import parmed
from sklearn.cluster import DBSCAN
from MDAnalysis.analysis import contacts
from MDAnalysis.analysis import distances
from collections import Counter
import nglview as nv
import MDAnalysis as mda
import numpy as np
from MDAnalysis.transformations import wrap, center_in_box
from scipy.spatial.distance import pdist,squareform
import seaborn as sns
import pytim
import warnings
import pandas as pd
import matplotlib.pyplot as plt

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from MDAnalysis import transformations as trans
import os
from scipy import sparse
import os
import fnmatch as fm
import numpy as np
import math
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import parmed
import numpy as np
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt

In [ ]:



### For RNA G and A, it is 13
### For RNA U and C, it is 11.5

def clustering(u, frame,eps=13):

        ### Function that is nested in all of the analysis methods in this file ###

        ca_sel = u.select_atoms("name C1' or name CA")

        cas = [atom.index for atom in ca_sel.atoms]

        to_cluster = [atom for i, atom in enumerate(u.trajectory[frame.frame].positions)if i in cas]

        n_ca = len(ca_sel)

        self_distances = distances.self_distance_array(ca_sel.positions,
                                                       box=frame.dimensions)

        sq_dist_arr = np.zeros((n_ca, n_ca))
        triu = np.triu_indices_from(sq_dist_arr, k=1)

        sq_dist_arr[triu] = self_distances
        sq_dist_arr.T[triu] = self_distances
  
        model = DBSCAN(eps=eps, min_samples=1, metric='precomputed')
    
        labels = model.fit_predict(sq_dist_arr)
        b = Counter(labels)

        clustered_list = [atom for j, atom in enumerate(to_cluster)
                          for i, item in enumerate(labels)
                          if item == b.most_common()[0][0]
                          and j == i]

        centroid = np.mean(clustered_list, axis=0)

        ind_list = [cas[index] for index, j in enumerate(to_cluster)
                    for i in clustered_list
                    if j[0]==i[0] and j[1]==i[1] and j[2]==i[2]]

        index_list = [x+1 for x in ind_list]

        res_list = [u.atoms[i].resindex for i in index_list]

        return centroid, to_cluster, clustered_list, res_list

def visu3(u):
    view=nv.show_mdanalysis(u)
    view.clear_representations()
    view.add_representation('surface',sele='protein',opacity=0.8,color='green')
    view.add_representation('ball+stick',sele='nucleic',aspectRatio=2)
    view.add_representation('surface',sele='LYS',opacity=0.8,color='green')
    #view.add_representation('ball+stick',sele='LYS',aspectRatio=2)
    view.add_representation('surface',sele='nucleic',opacity=0.8)
    view.add_representation('ball+stick',sele='ions',aspectRatio=3)
    #view.add_representation('ball+stick',sele='water',aspectRatio=2)
    view.add_unitcell(color='white')
    return view


def visul5(unive,s1,s2):
    view=nv.show_mdanalysis(unive)
    view.clear_representations()
    view.add_representation('ball+stick',sele=f"(nucleic) and ( {s1})",
                            opacity=1.5,aspectRatio=2,color='red')  
    
    view.add_representation('ball+stick',sele=f"(protein) and ( {s1})",
                            opacity=1.5,aspectRatio=2,color='yellow')  
    

    view.add_representation('ball+stick',sele=f"(nucleic or protein ) and ({s2})",
                            opacity=1.5,
                            aspectRatio=2,
                            color='green')

    view.add_unitcell(color='white')
    return view


def visul6(unive,s1,s2):
    view=nv.show_mdanalysis(unive)
    view.clear_representations()
    view.add_representation('ball+stick',sele=f"(nucleic) and ( {s1})",
                            opacity=1.5,aspectRatio=2,color='red')  
    
    view.add_representation('ball+stick',sele=f"(protein) and ( {s1})",
                            opacity=1.5,aspectRatio=2,color='yellow')  
    

    view.add_representation('ball+stick',sele=f"(SOL ) and ({s2})",
                            opacity=1.5,
                            aspectRatio=8,
                            color='green')
    
    

    view.add_unitcell(color='white')
    return view


def visul7(unive,s1,s2,s3):
    view=nv.show_mdanalysis(unive)
    view.clear_representations()
    view.add_representation('ball+stick',sele=f"(nucleic) and ( {s1})",
                            opacity=1.5,aspectRatio=2,color='red')  
    
    
    view.add_representation('ball+stick',sele=f"(protein) and ( {s1})",
                            opacity=1.5,aspectRatio=4,color='yellow')  
    

    view.add_representation('ball+stick',sele=f"(protein) and ({s2})",
                            opacity=1.5,
                            aspectRatio=4,
                            color='green')

    view.add_representation('ball+stick',sele=f"(protein) and ({s3})",
                            opacity=1.5,
                            aspectRatio=4,
                            color='red')
    
    

    view.add_unitcell(color='white')
    return view


def visul8(unive,s1,s2,s3,water):
    view=nv.show_mdanalysis(unive)
    view.clear_representations()
    view.add_representation('ball+stick',sele=f"(nucleic) and ( {s1})",
                            opacity=0.5,aspectRatio=2,color='red')  
    
    
    view.add_representation('ball+stick',sele=f"(protein) and ( {s1})",
                            opacity=0.5,aspectRatio=4,color='yellow')  
    

    view.add_representation('ball+stick',sele=f"(protein) and ({s2})",
                            opacity=0.5,
                            aspectRatio=4,
                            color='green')

    view.add_representation('ball+stick',sele=f"(protein) and ({s3})",
                            opacity=0.5,
                            aspectRatio=4,
                            color='red')
    
    view.add_representation('ball+stick',sele=f"{water}",
                            opacity=1.5,
                            aspectRatio=4,
                            color='orange')
        

    view.add_unitcell(color='white')
    return view

In [ ]:

def surface_comp2(u, start=4000, stop=-1, step=10000):
    frame_num = 0
    inner_layers_all = []
    outer_layers_all = []

    for frame in u.trajectory[start:stop:step]:
        centroid, to_cluster, clustered_list, res_list = clustering(u, frame, eps=9)

        # --- Build selection for clustered core residues ---
        selection = 'resindex '
        for item in res_list:
            selection += str(item) + ' or resindex '
        selection = selection[0:-13]

        sel = u.select_atoms(selection)

        # --- Write clustered residues to temp PDB and reload ---
        with mda.Writer('spatial.pdb') as pdb:
            pdb.write(sel)
        univ = mda.Universe('spatial.pdb')

        # --- Run GITIM surface analysis ---
        g = univ.select_atoms('name CA')
        interface = pytim.GITIM(
            univ,
            group=g,
            alpha=4.0,
            max_layers=3,
            molecular=False,
            centered='middle'
        )

        # --- Extract layer 1 (outer) and layer 2 (inner) via interface object ---
        layer1_atoms = interface.layers[0]   # outer surface
        layer2_atoms = interface.layers[1]   # mid layer
        layer3_atoms = interface.layers[2]   # inner layer

        outer_resids = np.unique(layer1_atoms.resids)
        mid_resids = np.unique(layer2_atoms.resids)
        inner_resids = np.unique(layer3_atoms.resids)
        
        
        return inner_resids ,mid_resids,outer_resids,res_list
    
    
    
    
# 1. Define the model function
def constrained_exp(t, A, T1, c):
    T2 = c * T1
    return A * np.exp(-t / T1) + (1 - A) * np.exp(-t / T2)

# 2. Refined fit function to return Teff
def fit_constrained(sp, label):
    t = np.arange(len(sp))
    y = np.array(sp)
    
    # normalize
    y = y / y[0]
    
    # initial guesses and bounds
    p0 = [0.5, 10.0, 5.0]  # A, T1, c
    bounds = ([0, 0, 1], [1, np.inf, np.inf])
    
    popt, _ = curve_fit(constrained_exp, t, y, p0=p0, bounds=bounds)
    
    A, T1, c = popt
    T2 = c * T1
    tau_avg = A * T1 + (1 - A) * T2
    
    # Plotting
    plt.plot(t, y, 'o', alpha=0.3, label=f'{label} data')
    plt.plot(t, constrained_exp(t, *popt), '-', linewidth=2, label=f'{label} fit')
    
    return tau_avg


def SP(univ, start, stop, select):
    dist_water = 5.0

    # --- go to initial frame ---
    univ.trajectory[start]

    # --- initial selection ---
    initial_sel = univ.select_atoms(
        f"name OW and (around {dist_water} (protein and resid {select}))"
    )

    # --- use indices (BEST choice) ---
    initial_ids = set(initial_sel.indices)

    print(f"Initial waters: {len(initial_ids)}")

    # --- initialize ---
    alive_ids = set(initial_ids)

    frames = []
    survival_counts = []
    survival_fraction = []

    # --- loop ---
    for ts in univ.trajectory[start:stop]:

        current_sel = univ.select_atoms(
            f"name OW and (around {dist_water} (protein and resid {select}))"
        )

        current_ids = set(current_sel.indices)

        # --- enforce continuity + initial membership ---
        alive_ids = alive_ids.intersection(current_ids)

        count_surviving = len(alive_ids)

        frames.append(ts.frame)
        survival_counts.append(count_surviving)
        survival_fraction.append(count_surviving / len(initial_ids))

    return survival_fraction



def surface_comp3(u, start=4000, stop=-1, step=10000):
    
    frame_num = 0
    inner_layers_all = []
    outer_layers_all = []

    for frame in u.trajectory[start:stop:step]:
        centroid, to_cluster, clustered_list, res_list = clustering(u, frame, eps=9)

        # --- Build selection for clustered core residues ---
        selection = 'resindex '
        for item in res_list:
            selection += str(item) + ' or resindex '
        selection = selection[0:-13]

        sel = u.select_atoms(selection)

        # --- Write clustered residues to temp PDB and reload ---
        with mda.Writer('spatial.pdb') as pdb:
            pdb.write(sel)
        univ = mda.Universe('spatial.pdb')

        # --- Run GITIM surface analysis ---
        g = univ.select_atoms('name CA')
        interface = pytim.GITIM(
            univ,
            group=g,
            alpha=4.0,
            max_layers=2,
            molecular=False,
            centered='middle'
        )

        # --- Extract layer 1 (outer) and layer 2 (inner) via interface object ---
        layer1_atoms = interface.layers[0]   # outer surface
        layer2_atoms = interface.layers[1]   # inner layer

        outer_resids = np.unique(layer1_atoms.resids)
        inner_resids = np.unique(layer2_atoms.resids)
        
        
        return inner_resids, outer_resids, res_list
    
    

def SP_2(univ, start, stop, select, select_2):
    dist_water_inner = 3.5
    dist_water_outer = 2.9

    # --- go to initial frame ---
    univ.trajectory[start]

    # --- initial selection ---
    inner_water = univ.select_atoms(
        f"name OW and (around {dist_water_inner} (protein and resid {select}))"
    )
    outer_water = univ.select_atoms(
        f"name OW and (around {dist_water_outer} (protein and resid {select_2}))"
    )

    # --- remove waters that appear in both shells ---
    
    inner_ids = set(inner_water.indices)
    outer_ids = set(outer_water.indices)

    common_ids = inner_ids.intersection(outer_ids)
    inner_ids = inner_ids - common_ids  # exclusive inner shell
#    outer_ids = outer_ids - common_ids  # exclusive outer shell

    print(f"Initial inner waters: {len(inner_ids)}")
    print(f"Initial outer waters: {len(outer_ids)}")

    # --- initialize survival tracking separately ---
    alive_inner_ids = set(inner_ids)
    alive_outer_ids = set(outer_ids)

    frames = []
    survival_counts_inner = []
    survival_counts_outer = []
    survival_fraction_inner = []
    survival_fraction_outer = []

    # --- loop over trajectory ---
    for ts in univ.trajectory[start:stop]:
        # current waters near inner residue
        current_inner_sel = univ.select_atoms(
            f"name OW and (around {dist_water_inner} (protein and resid {select}))"
        )
        # current waters near outer residue
        current_outer_sel = univ.select_atoms(
            f"name OW and (around {dist_water_outer} (protein and resid {select_2}))"
        )

        current_inner_ids = set(current_inner_sel.indices)
        current_outer_ids = set(current_outer_sel.indices)

        # --- enforce continuity: keep only originally selected waters still present ---
        alive_inner_ids = alive_inner_ids.intersection(current_inner_ids)
        alive_outer_ids = alive_outer_ids.intersection(current_outer_ids)

        frames.append(ts.frame)

        survival_counts_inner.append(len(alive_inner_ids))
        survival_counts_outer.append(len(alive_outer_ids))

        # guard against division by zero
        survival_fraction_inner.append(
            len(alive_inner_ids) / len(inner_ids) if inner_ids else 0.0
        )
        survival_fraction_outer.append(
            len(alive_outer_ids) / len(outer_ids) if outer_ids else 0.0
        )

    return survival_fraction_inner, survival_fraction_outer

In [ ]:

# define functions
from scipy import optimize
from scipy import sparse

def timeCorr_bool(mats, N, m):
    # mats is a list of bool matrix, which express the bindings
    # N= len(mats)
    # m: time region of correlation, should < N/2
    corr_m= np.zeros(m)
    #cnt= 1
    for i in range(m):
        # matCorr= []
        matCorr= np.zeros(N-i)
        for k in range(N-i):
            matCorr[k]= (mats[k] & mats[k+i]).sum()
            # matCorr.append( (mats[k] * mats[k+i]).sum() )
        corr_m[i]= np.mean(matCorr)
        
        '''if i>= ((cnt*m)//25)-1:
            print(f"{cnt*25} %")
            cnt+=1'''
        
    corrN_m= corr_m/corr_m[0]
    return corrN_m, corr_m

def timeCorr_boolCsr(mats, N, m):
    # mats is a list of bool csr_matrix, which express the bindings
    # N= len(mats)
    # m: time region of correlation, should < N/2
    corr_m= np.zeros(m)
    # cnt= 1
    for i in range(m):
        # matCorr= []
        matCorr= np.zeros(N-i)
        for k in range(N-i):
            matCorr[k]= (mats[k].multiply(mats[k+i])).sum()
        corr_m[i]= np.mean(matCorr)
        
        '''if i>= ((cnt*m)//10)-1:
            print(f"{cnt*10} %")
            cnt+=1'''
        
    corrN_m= corr_m/corr_m[0]
    return corrN_m, corr_m

def exp2(t, a, tau1, c):
    return a * np.exp(-t / tau1) + (1 - a) * np.exp(-t / (c*tau1))

def fit_corr(Ct, t, init):
    # fit normalized correlation ( C(0)=1 )
    n= len(t)
    popt2, pcov2 = optimize.curve_fit(
            exp2, t, Ct, sigma= (t+t[1])**(1/3.0),
            bounds=([0,0,1],[1,10*t[-1],math.inf]), p0=init, maxfev=1000000) # sigma= t**0.5,
    
    Ct_fit= np.zeros(n)
    for i in range(n):
        Ct_fit[i]= exp2(t[i], *popt2)
    return Ct_fit, popt2    

def calc_contact(grp1, grp2, d, dims):
    # calculate contact matrix
    dist= contacts.distance_array(grp1.positions, grp2.positions, box=dims)
    contMat_12= contacts.contact_matrix(dist, radius=d)
    return contMat_12


def timeCorr_boolCsr_fromStart(mats, N, m):
    # mats is a list of bool csr_matrix, which express the bindings
    # N= len(mats)
    # m: time region of correlation, should < N/2
    corr_m = np.zeros(m)
    
    for i in range(m):
        corr_m[i] = (mats[0].multiply(mats[i])).sum()
    
    corrN_m = corr_m / corr_m[0]
    return corrN_m, corr_m


### Water - condensate correlation 

In [ ]:

systems    = ['2_close', '2_spread', '3_close', '3_spread']

systems    = ['2_close']

clrs       = ['#E01010', '#1010E0', '#10A010', '#A010A0', '#10A0A0']
clrs2      = ['#E09010', '#1090E0', '#10C040', '#C040C0', '#40C0C0']

d_cont     = 5
star       = 1000
m_corr     = 500
dt         = 2e-3
eps_clust  = 9

offsets    = [0]

save_dir   = './corr_results'
os.makedirs(save_dir, exist_ok=True)

results = {}

for idx, label in enumerate(systems):

    print(f"\n{'='*60}")
    print(f"  Processing: {label}")
    print(f"{'='*60}")

    # ── Load universe ──────────────────────────────────────────────────────────
    stru = parmed.load_file(f'../gxg_comps/9_res/equal/{label}/output.top')
    u = mda.Universe(stru, f'../gxg_comps/9_res/equal/{label}/traj.dcd')

    a        = u.atoms
    workflow = [trans.wrap(a), trans.unwrap(a)]
    u.trajectory.add_transformations(*workflow)

    tM = dt * np.arange(m_corr)

    all_corrN    = []
    all_corr     = []
    all_corrN_2  = []   # FIX: initialize outer-layer lists
    all_corr_2   = []

    for number in offsets:
        

        traj_start = -(star + number)
        traj_stop  = len(u.trajectory)   # FIX: define frame_num from trajectory length
        print(f"\n  --- Offset {number} (traj slice [{traj_start}:]) ---")

        # --- Compute layers ---  FIX: use u, not univ; use traj_stop
        inner_layers, outer_layers, res_list = surface_comp3(
            u, start=traj_start, stop=traj_start+1, step=1
        )

        dist_water_inner = 3.1
        dist_water_outer = 3

        inner_layer_list = ' '.join(str(item) for item in inner_layers)
        outer_layer_list = ' '.join(str(item) for item in outer_layers)

        # --- Select waters ---  FIX: use u, not univ
        inner_water = u.select_atoms(
            f"name OW and (around {dist_water_inner} (protein and resid {inner_layer_list}))"
        )
        outer_water = u.select_atoms(
            f"name OW and (around {dist_water_outer} (protein and resid {outer_layer_list}))"
        )

        # --- Remove overlapping waters ---
        inner_ids  = set(inner_water.indices)
        outer_ids  = set(outer_water.indices)
        common_ids = inner_ids.intersection(outer_ids)
        inner_ids  = inner_ids - common_ids

        # ── Inner layer correlation ────────────────────────────────────────────
        idp_1 = u.select_atoms(f"(name N* or name O*) and (resid {inner_layer_list})")[::2]

        inner_water_str = ' '.join(str(item) for item in inner_ids)
        inner_water_sel = u.select_atoms(f"index {inner_water_str}")

        print(f"    idp_1: {len(idp_1)} atoms  |  inner_water: {len(inner_water_sel)} atoms")
        
        

        contMt = []
        for _ in u.trajectory[traj_start:traj_start+500]:
            contMat = calc_contact(idp_1, inner_water_sel, d_cont, u.dimensions)
            contMt.append(sparse.csr_matrix(contMat))

        print(f"    Inner contact matrix done. ({len(contMt)} frames)")

        corrN, corr = timeCorr_boolCsr_fromStart(contMt, len(contMt), m_corr)
        all_corrN.append(corrN)
        all_corr.append(corr)

        # ── Outer layer correlation ────────────────────────────────────────────
        idp_2 = u.select_atoms(f"(name N* or name O*) and (resid {outer_layer_list})")[::2]

        outer_water_str = ' '.join(str(item) for item in outer_ids)
        outer_water_sel = u.select_atoms(f"index {outer_water_str}")

        print(f"    idp_2: {len(idp_2)} atoms  |  outer_water: {len(outer_water_sel)} atoms")  # FIX: idp_2, outer_water

        contMt_2 = []
        for _ in u.trajectory[traj_start:traj_start+500]:
            contMat_2 = calc_contact(idp_2, outer_water_sel, d_cont, u.dimensions)
            contMt_2.append(sparse.csr_matrix(contMat_2))

        print(f"    Outer contact matrix done. ({len(contMt_2)} frames)")   # FIX: contMt_2

        corrN_2, corr_2 = timeCorr_boolCsr_fromStart(contMt_2, len(contMt_2), m_corr)
        all_corrN_2.append(corrN_2)
        all_corr_2.append(corr_2)

    # ── Average across offsets — inner ─────────────────────────────────────────
    corrN_avg   = np.mean(np.stack(all_corrN,   axis=0), axis=0)
    corr_avg    = np.mean(np.stack(all_corr,    axis=0), axis=0)
    corrN_std   = np.std( np.stack(all_corrN,   axis=0), axis=0)

    # ── Average across offsets — outer ─────────────────────────────────────────  FIX: was missing
    corrN_avg_2 = np.mean(np.stack(all_corrN_2, axis=0), axis=0)
    corr_avg_2  = np.mean(np.stack(all_corr_2,  axis=0), axis=0)
    corrN_std_2 = np.std( np.stack(all_corrN_2, axis=0), axis=0)

    print(f"\n  Averaged over {len(offsets)} offsets: {offsets}")

    # ── Exponential fitting — inner ────────────────────────────────────────────
    corrFit, para = fit_corr(corrN_avg, tM, [0.5, tM[-1] / 4, 10])
    t1    = para[1]
    t2    = para[1] * para[2]
    t_eff = para[0] * para[1] + (1 - para[0]) * t2
    print(f"  Inner — τ1={t1:.3f} ns, τ2={t2:.3f} ns, τ_eff={t_eff:.3f} ns")

    # ── Exponential fitting — outer ────────────────────────────────────────────  FIX: was missing
    corrFit_2, para_2 = fit_corr(corrN_avg_2, tM, [0.5, tM[-1] / 4, 10])
    t1_2    = para_2[1]
    t2_2    = para_2[1] * para_2[2]
    t_eff_2 = para_2[0] * para_2[1] + (1 - para_2[0]) * t2_2
    print(f"  Outer — τ1={t1_2:.3f} ns, τ2={t2_2:.3f} ns, τ_eff={t_eff_2:.3f} ns")

    # ── Store results ──────────────────────────────────────────────────────────
    results[label] = {
        # inner
        'tM':          tM,
        'corrN':       corrN_avg,
        'corrN_std':   corrN_std,
        'corr':        corr_avg,
        'all_corrN':   all_corrN,
        'all_corr':    all_corr,
        'corrFit':     corrFit,
        'para':        para,
        't1':          t1,
        't2':          t2,
        't_eff':       t_eff,
        # outer  FIX: was entirely missing
        'corrN_2':     corrN_avg_2,
        'corrN_std_2': corrN_std_2,
        'corr_2':      corr_avg_2,
        'all_corrN_2': all_corrN_2,
        'all_corr_2':  all_corr_2,
        'corrFit_2':   corrFit_2,
        'para_2':      para_2,
        't1_2':        t1_2,
        't2_2':        t2_2,
        't_eff_2':     t_eff_2,
        # shared
        'offsets':     offsets,
    }

# ── Save ───────────────────────────────────────────────────────────────────────
save_path = os.path.join(save_dir, 'water_corr_results.pkl')
with open(save_path, 'wb') as f:
    pickle.dump(results, f)
print(f"\n  Results saved to {save_path}")

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os

# ── Load results ───────────────────────────────────────────────────────────────
load_path = './corr_results/water_corr_resultsi_9res_2.pkl'
with open(load_path, 'rb') as f:
    results = pickle.load(f)

print(f"Systems loaded: {list(results.keys())}")

clrs  = ['#E01010', '#1010E0', '#10A010', '#A010A0', '#10A0A0']
clrs2 = ['#E09010', '#1090E0', '#10C040', '#C040C0', '#40C0C0']

# ── Plot ───────────────────────────────────────────────────────────────────────
n_sys = len(results)
fig, axes = plt.subplots(n_sys, 2, figsize=(14, 4 * n_sys), sharex=False)

# ensure axes is always 2D
if n_sys == 1:
    axes = axes[np.newaxis, :]

for idx, (label, res) in enumerate(results.items()):

    tM        = res['tM']

    # ── inner ──────────────────────────────────────────────────────────────────
    corrN     = res['corrN']
    corrN_std = res['corrN_std']
    corrFit   = res['corrFit']
    t1        = res['t1']
    t2        = res['t2']
    t_eff     = res['t_eff']

    # ── outer ──────────────────────────────────────────────────────────────────
    corrN_2     = res['corrN_2']
    corrN_std_2 = res['corrN_std_2']
    corrFit_2   = res['corrFit_2']
    t1_2        = res['t1_2']
    t2_2        = res['t2_2']
    t_eff_2     = res['t_eff_2']

    # ── per-offset traces (faint) ──────────────────────────────────────────────
    for off_corr in res['all_corrN']:
        axes[idx, 0].plot(tM, off_corr, color=clrs[idx], alpha=0.15, lw=0.8)
    for off_corr in res['all_corrN_2']:
        axes[idx, 1].plot(tM, off_corr, color=clrs2[idx], alpha=0.15, lw=0.8)

    # ── averaged data + std band ───────────────────────────────────────────────
    axes[idx, 0].fill_between(tM, corrN - corrN_std, corrN + corrN_std,
                               color=clrs[idx], alpha=0.2)
    axes[idx, 0].plot(tM, corrN, color=clrs[idx], lw=1.8, label='avg data')

    axes[idx, 1].fill_between(tM, corrN_2 - corrN_std_2, corrN_2 + corrN_std_2,
                               color=clrs2[idx], alpha=0.2)
    axes[idx, 1].plot(tM, corrN_2, color=clrs2[idx], lw=1.8, label='avg data')

    # ── exponential fit ────────────────────────────────────────────────────────
    axes[idx, 0].plot(tM, corrFit, 'k--', lw=1.5,
                      label=f'fit\nτ1={t1:.3f} ns\nτ2={t2:.3f} ns\nτ_eff={t_eff:.3f} ns')
    axes[idx, 1].plot(tM, corrFit_2, 'k--', lw=1.5,
                      label=f'fit\nτ1={t1_2:.3f} ns\nτ2={t2_2:.3f} ns\nτ_eff={t_eff_2:.3f} ns')

    # ── formatting ─────────────────────────────────────────────────────────────
    for col, layer in enumerate(['Inner layer', 'Outer layer']):
        axes[idx, col].set_xlabel('Time (ns)', fontsize=12)
        axes[idx, col].set_ylabel('C(t)', fontsize=12)
        axes[idx, col].set_title(f'{label} — {layer}', fontsize=13, fontweight='bold')
        axes[idx, col].set_xlim(tM[0], tM[-1])
        axes[idx, col].set_ylim(-0.05, 1.05)
        axes[idx, col].legend(fontsize=9, loc='upper right')
        axes[idx, col].grid(True, alpha=0.3, linestyle='--')
        axes[idx, col].tick_params(labelsize=10)

plt.tight_layout()

save_path = './corr_results/water_corr_fit_plot.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f"Plot saved to {save_path}")
plt.show()

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt

# ── Load ───────────────────────────────────────────────────────────────────────
with open('./corr_results/water_corr_resultsi_9res.pkl', 'rb') as f:
    results = pickle.load(f)

clrs  = ['#E01010', '#1010E0', '#10A010', '#A010A0', '#10A0A0']
clrs2 = ['#E09010', '#1090E0', '#10C040', '#C040C0', '#40C0C0']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].set_title('Inner Layer', fontsize=13, fontweight='bold')
axes[1].set_title('Outer Layer', fontsize=13, fontweight='bold')

for idx, (label, res) in enumerate(results.items()):
    tM = res['tM']

    # ── inner ──────────────────────────────────────────────────────────────────
#     axes[0].fill_between(tM,
#                          res['corrN'] - res['corrN_std'],
#                          res['corrN'] + res['corrN_std'],
#                          color=clrs[idx], alpha=0.15)
    axes[0].plot(tM, res['corrN'],   color=clrs[idx],  lw=1.8, label=label)
    axes[0].plot(tM, res['corrFit'], color=clrs[idx],  lw=1.5, ls='--',
                 label=f"{label} fit (τ_eff={res['t_eff']:.3f} ns)")

    # ── outer ──────────────────────────────────────────────────────────────────
    axes[1].fill_between(tM,
                         res['corrN_2'] - res['corrN_std_2'],
                         res['corrN_2'] + res['corrN_std_2'],
                         color=clrs2[idx], alpha=0.15)
    axes[1].plot(tM, res['corrN_2'],   color=clrs2[idx], lw=1.8, label=label)
    axes[1].plot(tM, res['corrFit_2'], color=clrs2[idx], lw=1.5, ls='--',
                 label=f"{label} fit (τ_eff={res['t_eff_2']:.3f} ns)")

for ax in axes:
    ax.set_xlabel('Time (ns)', fontsize=12)
    ax.set_ylabel('C(t)',      fontsize=12)
    ax.set_xlim(tM[0], tM[-1])
    ax.set_ylim(-0.05, 1.05)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.tick_params(labelsize=10)

plt.suptitle('Water–Protein Contact Correlation', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()

save_path = './corr_results/water_corr_comparison.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f"Saved: {save_path}")
plt.show()

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt



# ── Load ───────────────────────────────────────────────────────────────────────
with open('./corr_results/water_corr_resultsi_9res_2.pkl', 'rb') as f:
    results = pickle.load(f)

clrs  = ['#E01010', '#1010E0', '#10A010', '#A010A0', '#10A0A0']
clrs2 = ['#E09010', '#1090E0', '#10C040', '#C040C0', '#40C0C0']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].set_title('Inner Layer', fontsize=13, fontweight='bold')
axes[1].set_title('Outer Layer', fontsize=13, fontweight='bold')

for idx, (label, res) in enumerate(results.items()):
    tM = res['tM']

    # ── inner ──────────────────────────────────────────────────────────────────
#     axes[0].fill_between(tM,
#                          res['corrN'] - res['corrN_std'],
#                          res['corrN'] + res['corrN_std'],
#                          color=clrs[idx], alpha=0.15)
    axes[0].plot(tM, res['corrN'],   color=clrs[idx],  lw=1.8, label=label)
    axes[0].plot(tM, res['corrFit'], color=clrs[idx],  lw=1.5, ls='--',
                 label=f"{label} fit (τ_eff={res['t_eff']:.3f} ns)")

    # ── outer ──────────────────────────────────────────────────────────────────
    axes[1].fill_between(tM,
                         res['corrN_2'] - res['corrN_std_2'],
                         res['corrN_2'] + res['corrN_std_2'],
                         color=clrs2[idx], alpha=0.15)
    axes[1].plot(tM, res['corrN_2'],   color=clrs2[idx], lw=1.8, label=label)
    axes[1].plot(tM, res['corrFit_2'], color=clrs2[idx], lw=1.5, ls='--',
                 label=f"{label} fit (τ_eff={res['t_eff_2']:.3f} ns)")

for ax in axes:
    ax.set_xlabel('Time (ns)', fontsize=12)
    ax.set_ylabel('C(t)',      fontsize=12)
    ax.set_xlim(tM[0], tM[-1])
    ax.set_ylim(-0.05, 1.05)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.tick_params(labelsize=10)

plt.suptitle('Water–Protein Contact Correlation', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()

save_path = './corr_results/water_corr_comparison.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f"Saved: {save_path}")
plt.show()

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt



# ── Load ───────────────────────────────────────────────────────────────────────
with open('./corr_results/water_corr_resultsi_3res.pkl', 'rb') as f:
    results = pickle.load(f)

clrs  = ['#E01010', '#1010E0', '#10A010', '#A010A0', '#10A0A0']
clrs2 = ['#E09010', '#1090E0', '#10C040', '#C040C0', '#40C0C0']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].set_title('Inner Layer', fontsize=13, fontweight='bold')
axes[1].set_title('Outer Layer', fontsize=13, fontweight='bold')

for idx, (label, res) in enumerate(results.items()):
    tM = res['tM']

    # ── inner ──────────────────────────────────────────────────────────────────
#     axes[0].fill_between(tM,
#                          res['corrN'] - res['corrN_std'],
#                          res['corrN'] + res['corrN_std'],
#                          color=clrs[idx], alpha=0.15)
    axes[0].plot(tM, res['corrN'],   color=clrs[idx],  lw=1.8, label=label)
    axes[0].plot(tM, res['corrFit'], color=clrs[idx],  lw=1.5, ls='--',
                 label=f"{label} fit (τ_eff={res['t_eff']:.3f} ns)")

    # ── outer ──────────────────────────────────────────────────────────────────
    axes[1].fill_between(tM,
                         res['corrN_2'] - res['corrN_std_2'],
                         res['corrN_2'] + res['corrN_std_2'],
                         color=clrs2[idx], alpha=0.15)
    axes[1].plot(tM, res['corrN_2'],   color=clrs2[idx], lw=1.8, label=label)
    axes[1].plot(tM, res['corrFit_2'], color=clrs2[idx], lw=1.5, ls='--',
                 label=f"{label} fit (τ_eff={res['t_eff_2']:.3f} ns)")

for ax in axes:
    ax.set_xlabel('Time (ns)', fontsize=12)
    ax.set_ylabel('C(t)',      fontsize=12)
    ax.set_xlim(tM[0], tM[-1])
    ax.set_ylim(-0.05, 1.05)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.tick_params(labelsize=10)

plt.suptitle('Water–Protein Contact Correlation', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()

save_path = './corr_results/water_corr_comparison.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f"Saved: {save_path}")
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  Main analysis loop
# ══════════════════════════════════════════════════════════════════════════════

systems    = ['2_close', '2_spread', '3_close', '3_spread']

systems    = ['2_close']

clrs       = ['#E01010', '#1010E0', '#10A010', '#A010A0', '#10A0A0']
clrs2      = ['#E09010', '#1090E0', '#10C040', '#C040C0', '#40C0C0']

# ── Parameters ────────────────────────────────────────────────────────────────
d_cont     = 5        # Å  contact cutoff
star       = 2000     # frames to analyse
m_corr     = 2000     # correlation window  (~2830)
dt         = 2e-3     # ps per frame → ns  (2 ps = 0.002 ns)
eps_clust  = 9        # DBSCAN eps for clustering

offsets    = [0,1000, 2000, 3000, 4000]   # multiple trajectory start offsets

save_dir   = './corr_results'
os.makedirs(save_dir, exist_ok=True)

results = {}

for idx, label in enumerate(systems):

    print(f"\n{'='*60}")
    print(f"  Processing: {label}")
    print(f"{'='*60}")

    # ── Load universe ─────────────────────────────────────────────────────────
    stru = parmed.load_file(f'../gxg_comps/9_res/equal/{label}/output.top')
    #stru.strip('@OW,HW1,HW2')
    u = mda.Universe(stru, f'../gxg_comps/9_res/equal/{label}/traj.dcd')

    a        = u.atoms
    workflow = [trans.wrap(a), trans.unwrap(a)]
    u.trajectory.add_transformations(*workflow)

    # ── Time axis (in ns) — shared across all offsets ─────────────────────────
    tM = dt * np.arange(m_corr)   # ns

    # ── Multi-offset loop: clustering + contacts + correlation ────────────────
    all_corrN  = []   # normalised correlations, one per offset
    all_corr   = []   # raw correlations, one per offset

    for number in offsets:
        traj_start = -(star + number)
        print(f"\n  --- Offset {number} (traj slice [{traj_start}:]) ---")
        
        # --- Compute layers ---
        inner_layers, outer_layers, res_list = surface_comp3(
            univ, start=traj_start, stop=frame_num+1, step=1
        )

        dist_water_inner = 3
        dist_water_outer = 3

        # Build selection strings
        inner_layer_list = ' '.join(str(item) for item in inner_layers)
        outer_layer_list = ' '.join(str(item) for item in outer_layers)

        # --- Select waters ---
        inner_water = univ.select_atoms(
            f"name OW and (around {dist_water_inner} (protein and resid {inner_layer_list}))"
        )
        outer_water = univ.select_atoms(
            f"name OW and (around {dist_water_outer} (protein and resid {outer_layer_list}))"
        )

        # --- Remove overlapping waters ---
        inner_ids = set(inner_water.indices)
        outer_ids = set(outer_water.indices)
        common_ids = inner_ids.intersection(outer_ids)
        inner_ids = inner_ids - common_ids  # exclusive inner shell
        outer_ids = outer_ids  # optional: exclusive outer shell        
        
        
#### Correlation for inner layer and water         

        idp_1 = u.select_atoms(f" (name N* or name O*) and (resid {inner_layer_list})")[::2]
    
        inner_water = ' '.join(str(item) for item in inner_ids)
        inner_water = u.select_atoms(f"index {inner_water}")
        
        print(f"    idp_1: {len(idp_1)} atoms  |  no_water: {len(inner_water)} atoms")

        
# ── Contact matrix accumulation ────────────────────────────────────
        contMt = []
    
        for _ in u.trajectory[traj_start:]:
            contMat = calc_contact(idp_1, inner_water, d_cont, u.dimensions)
            contMt.append(sparse.csr_matrix(contMat))

        print(f"    Contact matrix done. ({len(contMt)} frames)")
        
        # ── Time correlation ───────────────────────────────────────────────
        corrN, corr = timeCorr_boolCsr_fromStart(contMt, len(contMt), m_corr)

        all_corrN.append(corrN)
        all_corr.append(corr)
        
        
        
#####. Correlation for outer layer and water


        idp_2 = u.select_atoms(f" (name N* or name O*) and (resid {outer_layer_list})")[::2]
    
        outer_water = ' '.join(str(item) for item in outer_ids)
        outer_water = u.select_atoms(f"index {outer_water}")
        
        print(f"    idp_1: {len(idp_1)} atoms  |  no_water: {len(outer_water)} atoms")

        
# ── Contact matrix accumulation ────────────────────────────────────
        contMt_2 = []
    
        for _ in u.trajectory[traj_start:]:
            contMat_2 = calc_contact(idp_2, outer_water, d_cont, u.dimensions)
            contMt_2.append(sparse.csr_matrix(contMat_2))

        print(f"    Contact matrix done. ({len(contMt)} frames)")
        
        # ── Time correlation ───────────────────────────────────────────────
        corrN_2, corr_2 = timeCorr_boolCsr_fromStart(contMt_2, len(contMt_2), m_corr)

        all_corrN_2.append(corrN_2)
        all_corr_2.append(corr_2)
        

        
        
    # ── Average across offsets ─────────────────────────────────────────────────
    # Stack to (n_offsets, m_corr) then mean over axis 0
    corrN_avg = np.mean(np.stack(all_corrN, axis=0), axis=0)
    corr_avg  = np.mean(np.stack(all_corr,  axis=0), axis=0)
    corrN_std = np.std( np.stack(all_corrN, axis=0), axis=0)   # optional, useful for plotting

    print(f"\n  Averaged over {len(offsets)} offsets: {offsets}")

    # ── Exponential fitting on the averaged correlation ────────────────────────
    corrFit, para = fit_corr(corrN_avg, tM, [0.5, tM[-1] / 4, 10])
    t1    = para[1]
    t2    = para[1] * para[2]
    t_eff = para[0] * para[1] + (1 - para[0]) * t2
    print(f"  τ1 = {t1:.3f} ns,  τ2 = {t2:.3f} ns,  τ_eff = {t_eff:.3f} ns")

    # ── Store results ──────────────────────────────────────────────────────────
    results[label] = {
        'tM':        tM,
        'corrN':     corrN_avg,       # averaged normalised correlation
        'corrN_std': corrN_std,       # std across offsets (for error bars)
        'corr':      corr_avg,        # averaged raw correlation
        'all_corrN': all_corrN,       # per-offset normalised correlations
        'all_corr':  all_corr,        # per-offset raw correlations
        'offsets':   offsets,
        'corrFit':   corrFit,
        'para':      para,
        't1':        t1,
        't2':        t2,
        't_eff':     t_eff,
    }

# ── Save all results ───────────────────────────────────────────────────────────
save_path = os.path.join(save_dir, 'water_corr_results.pkl')
with open(save_path, 'wb') as f:
    pickle.dump(results, f)
print(f"\n  Results saved to {save_path}")










        
        


        residue_list = ' '
        for item in res_list:
            residue_list += str(item) + ' '

        # ── Atom selections (re-derived per offset) ────────────────────────
        idp_1 = u.select_atoms(f"name CA and (resid {residue_list})")[::3]
        idp_2 = u.select_atoms(f"name CA and (resid {residue_list})")

        # ── Intra-chain mask ───────────────────────────────────────────────
        resids_1   = idp_1.resids
        resids_2   = idp_2.resids
        chain_1    = resids_1 // chain_size
        chain_2    = resids_2 // chain_size
        intra_mask = (chain_1[:, None] == chain_2[None, :])

        print(f"    idp_1: {len(idp_1)} atoms  |  idp_2: {len(idp_2)} atoms  |  "
              f"chains: {chain_1.max() + 1}  |  dilute residues: {len(res_list)}")

        # ── Contact matrix accumulation ────────────────────────────────────
        contMt = []
        for _ in u.trajectory[traj_start:]:
            contMat = calc_contact(idp_1, idp_2, d_cont, u.dimensions)
            contMat[intra_mask] = False
            contMt.append(sparse.csr_matrix(contMat))

        print(f"    Contact matrix done. ({len(contMt)} frames)")

        # ── Time correlation ───────────────────────────────────────────────
        corrN, corr = timeCorr_boolCsr_fromStart(contMt, len(contMt), m_corr)

        all_corrN.append(corrN)
        all_corr.append(corr)

    # ── Average across offsets ─────────────────────────────────────────────────
    # Stack to (n_offsets, m_corr) then mean over axis 0
    corrN_avg = np.mean(np.stack(all_corrN, axis=0), axis=0)
    corr_avg  = np.mean(np.stack(all_corr,  axis=0), axis=0)
    corrN_std = np.std( np.stack(all_corrN, axis=0), axis=0)   # optional, useful for plotting

    print(f"\n  Averaged over {len(offsets)} offsets: {offsets}")

    # ── Exponential fitting on the averaged correlation ────────────────────────
    corrFit, para = fit_corr(corrN_avg, tM, [0.5, tM[-1] / 4, 10])
    t1    = para[1]
    t2    = para[1] * para[2]
    t_eff = para[0] * para[1] + (1 - para[0]) * t2
    print(f"  τ1 = {t1:.3f} ns,  τ2 = {t2:.3f} ns,  τ_eff = {t_eff:.3f} ns")

    # ── Store results ──────────────────────────────────────────────────────────
    results[label] = {
        'tM':        tM,
        'corrN':     corrN_avg,       # averaged normalised correlation
        'corrN_std': corrN_std,       # std across offsets (for error bars)
        'corr':      corr_avg,        # averaged raw correlation
        'all_corrN': all_corrN,       # per-offset normalised correlations
        'all_corr':  all_corr,        # per-offset raw correlations
        'offsets':   offsets,
        'corrFit':   corrFit,
        'para':      para,
        't1':        t1,
        't2':        t2,
        't_eff':     t_eff,
    }

# ── Save all results ───────────────────────────────────────────────────────────
save_path = os.path.join(save_dir, '9res_2_corr_results.pkl')
with open(save_path, 'wb') as f:
    pickle.dump(results, f)
print(f"\n  Results saved to {save_path}")

### Load data and compare

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

systems = ['charged', 'equal', 'aromatic', 'nonpolar', 'polar']
layers = ['Inner', 'Outer']

results = []

# ---- LOOP OVER SYSTEMS ----
for sys in systems:
    
    plt.figure()
    # Load both datasets
    try:
        df_9 = pd.read_csv(f'data/sp_9res_{sys}_new.csv')
        df_3 = pd.read_csv(f'data/sp_2res_{sys}_new.csv')
    except FileNotFoundError:
        print(f"Missing file for {sys}, skipping...")
        continue

    # ---- COMPUTE Teff ----
    teff_9 = {
        'Inner': fit_constrained(df_9['Inner'].values, 'Inner'),
        #'Mid':   fit_constrained(df_9['Mid'].values, 'Mid'),
        'Outer': fit_constrained(df_9['Outer'].values, 'Outer')
    }

    teff_3 = {
        'Inner': fit_constrained(df_3['Inner'].values, 'Inner'),
        #'Mid':   fit_constrained(df_3['Mid'].values, 'Mid'),
        'Outer': fit_constrained(df_3['Outer'].values, 'Outer')
    }

    # ---- STORE ----
    for layer in layers:
        results.append({
            'System': sys,
            'Layer': layer,
            'Teff_9res': teff_9[layer],
            'Teff_3res': teff_3[layer]
        })

# ---- CREATE DATAFRAME ----
summary_df = pd.DataFrame(results)

print("\n--- Combined Teff Comparison ---")
print(summary_df.to_string(index=False))

In [ ]:
summary_df

In [ ]:

for layer in layers:
    subset = summary_df[summary_df['Layer'] == layer]

    x = range(len(subset))

    plt.figure(figsize=(8,5))
    plt.bar(x, subset['Teff_9res'], width=0.4, label='9res', align='center')
    plt.bar([i + 0.4 for i in x], subset['Teff_3res'], width=0.4, label='3res')

    plt.xticks([i + 0.2 for i in x], subset['System'], rotation=45)
    plt.ylabel('Teff')
    plt.title(f'{layer} Layer: 3res vs 9res')
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
import matplotlib.pyplot as plt

layers = ['Inner', 'Mid', 'Outer']

fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey='row')

for col, layer in enumerate(layers):
    subset = summary_df[summary_df['Layer'] == layer]
    x = range(len(subset))

    # --- Row 1: 9res ---
    ax1 = axes[0, col]
    ax1.bar(x, subset['Teff_9res'], width=0.5)
    ax1.set_title(f"{layer} (9res)")
    ax1.set_xticks(x)
    ax1.set_xticklabels(subset['System'], rotation=45)
    ax1.set_ylabel('Teff')

    # --- Row 2: 3res ---
    ax2 = axes[1, col]
    ax2.bar(x, subset['Teff_3res'], width=0.5)
    ax2.set_title(f"{layer} (3res)")
    ax2.set_xticks(x)
    ax2.set_xticklabels(subset['System'], rotation=45)
    ax2.set_ylabel('Teff')

plt.tight_layout()
plt.show()
    

In [ ]:

# Optional: Plot the comparison of Teff across systems
summary_df.set_index('System').plot(kind='bar', figsize=(10, 6))
plt.ylabel('Effective Mean Residence Time (Teff)')
plt.title('Comparison of Teff across Chemical Systems and Layers')
plt.xticks(rotation=45)
plt.show()

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

systems = ['charged', 'equal', 'aromatic', 'nonpolar', 'polar']
layers = ['Inner', 'Mid', 'Outer']

for sys in systems:
        df_9 = pd.read_csv(f'data/sp_9res_{sys}_new.csv')
        df_3 = pd.read_csv(f'data/sp_2res_{sys}_new.csv')
        plt.figure(figsize=(7,5))
        
        for layer in layers:
            plt.plot(time, df_9[layer], label='9res', linewidth=2)
            plt.plot(time, df_3[layer], '--', label='3res', linewidth=2)

            plt.xlabel('Time')
            plt.ylabel('Survival Probability')
            plt.title(f'{sys.capitalize()} | {layer} Layer')
            plt.legend()
            plt.tight_layout()
        plt.show()        

        
        
#     except FileNotFoundError:
#         print(f"Missing {sys}")
#         continue

#     time = range(len(df_9))  # assuming same length

#     for layer in layers:
#         plt.figure(figsize=(7,5))



### 9_res

In [ ]:
# 3. Main Comparison Loop
systems = ['2_close', '2_spread', '3_close', '3_spread']


layers = ['Inner', 'Mid', 'Outer']
numbers = [60000, 70000, 80000, 90000]

for sys in systems:
    stru = parmed.load_file(f'../gxg_comps/9_res/{sys}/output.top',
                            xyz=f'../gxg_comps/9_res/{sys}/output.gro')
    univ = mda.Universe(stru, f'../gxg_comps/9_res/{sys}/traj.dcd')
    print(univ.trajectory)

    # Accumulate SP results across all windows
    all_inner = []
    all_mid   = []
    all_outer = []

    for num in numbers:
        # Identify layer compositions at the snapshot frame
        inner_layers, mid_layers, outer_layers, res_list = surface_comp2(
            univ, start=num, stop=num+1, step=1
        )

        # Build selection strings
        inner_layer_list = ' '.join(str(item) for item in inner_layers)
        mid_layer_list   = ' '.join(str(item) for item in mid_layers)
        outer_layer_list = ' '.join(str(item) for item in outer_layers)

        # Compute SP over the 800-frame window following the snapshot
        sp_inner = SP(univ, start=num, stop=num+800, select=inner_layer_list)
        sp_mid   = SP(univ, start=num, stop=num+800, select=mid_layer_list)
        sp_outer = SP(univ, start=num, stop=num+800, select=outer_layer_list)

        all_inner.append(sp_inner)
        all_mid.append(sp_mid)
        all_outer.append(sp_outer)

    # Average across all windows element-wise (assumes SP returns array-like)
    avg_inner = np.mean(all_inner, axis=0)
    avg_mid   = np.mean(all_mid,   axis=0)
    avg_outer = np.mean(all_outer, axis=0)

    # Save averaged results
    df = pd.DataFrame({
        'Inner': avg_inner,
        'Mid':   avg_mid,
        'Outer': avg_outer
    })
    df.to_csv(f'data/sp_9res_{sys}.csv', index=False)
    print(f"Data successfully saved for system: {sys}")

In [ ]:
# 3. Main Comparison Loop
systems = ['charged', 'equal', 'aromatic', 'nonpolar', 'polar']
systems = ['2_close', '2_spread', '3_close', '3_spread']
results = []

layers = ['Inner','Outer']


for lay in layers:
    plt.figure(figsize=(8, 5))
    for sys in systems:
        loaded_df = pd.read_csv(f'data/sp_9res_{sys}_new.csv')
    

for sys in systems:
    plt.figure(figsize=(8, 5))
    
    # Load data
    try:
        loaded_df = pd.read_csv(f'data/sp_9res_{sys}_new.csv')
    except FileNotFoundError:
        print(f"File for {sys} not found, skipping...")
        continue
        
    # Calculate Teff for each layer and plot
    teff_inner = fit_constrained(loaded_df['Inner'].values, 'Inner')
    #teff_mid   = fit_constrained(loaded_df['Mid'].values, 'Mid')
    teff_outer = fit_constrained(loaded_df['Outer'].values, 'Outer')

    # Store results for the final summary table
    results.append({
        'System': sys,
        'Inner_Teff': teff_inner,
        #'Mid_Teff': teff_mid,
        'Outer_Teff': teff_outer
    })

    plt.xlabel('Time')
    plt.ylabel('Normalized SP')
    plt.title(f'Constrained Double Exponential Fit - System: {sys.capitalize()}')
    plt.legend()
    plt.show()

# 4. Final Comparison Table
summary_df = pd.DataFrame(results)
print("\n--- Final Teff Comparison Summary ---")
print(summary_df.to_string(index=False))

# Optional: Plot the comparison of Teff across systems
summary_df.set_index('System').plot(kind='bar', figsize=(10, 6))
plt.ylabel('Effective Mean Residence Time (Teff)')
plt.title('Comparison of Teff across Chemical Systems and Layers')
plt.xticks(rotation=45)
plt.show()

In [ ]:
import seaborn as sns

# Assuming 'summary_df' was created in the previous step
# Melt the dataframe for easier plotting with Seaborn
melted_df = summary_df.melt(id_vars='System', var_name='Layer', value_name='Teff')

# Define the layers to plot separately
layers = ['Inner_Teff', 'Outer_Teff']

for layer_name in layers:
    plt.figure(figsize=(9, 5))
    
    # Filter data for the specific layer
    layer_data = summary_df[['System', layer_name]].sort_values(by=layer_name, ascending=False)
    
    # Create bar plot
    sns.barplot(x='System', y=layer_name, data=layer_data, palette='viridis')
    
    plt.title(f'Comparison of $T_{{eff}}$ across Systems: {layer_name.split("_")[0]} Layer')
    plt.ylabel('Mean Residence Time ($T_{eff}$)')
    plt.xlabel('Chemical System')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.show()

# Print the numeric breakdown
for layer in layers:
    print(f"\n--- Statistics for {layer} ---")
    print(summary_df[['System', layer]].sort_values(by=layer, ascending=False).to_string(index=False))

In [ ]:
# 3. Main Comparison Loop
systems = ['charged', 'polar', 'equal', 'aromatic', 'nonpolar']
#systems = ['2_close', '2_spread', '3_close', '3_spread']
results = []

layers = ['Inner','Mid','Outer']


for lay in layers:
    plt.figure(figsize=(4,3))
    for sys in systems:
        loaded_df = pd.read_csv(f'data/sp_9res_{sys}.csv')
        
        plt.plot(loaded_df[lay],label=sys)

    plt.legend()


### 3_res

In [ ]:
# 3. Main Comparison Loop

systems = ['charged', 'polar', 'equal', 'aromatic', 'nonpolar']


layers = ['Inner', 'Mid', 'Outer']
numbers = [100000, 110000,115000]

for sys in systems:
    stru = parmed.load_file(f'../gxg_comps/3_res/{sys}/output.top',
                            xyz=f'../gxg_comps/3_res/{sys}/output.gro')
    univ = mda.Universe(stru, f'../gxg_comps/3_res/{sys}/trajectory.dcd')
    print(univ.trajectory)

    # Accumulate SP results across all windows
    all_inner = []
    all_mid   = []
    all_outer = []

    for num in numbers:
        # Identify layer compositions at the snapshot frame
        inner_layers, mid_layers, outer_layers, res_list = surface_comp2(
            univ, start=num, stop=num+1, step=1
        )

        # Build selection strings
        inner_layer_list = ' '.join(str(item) for item in inner_layers)
        mid_layer_list   = ' '.join(str(item) for item in mid_layers)
        outer_layer_list = ' '.join(str(item) for item in outer_layers)

        # Compute SP over the 800-frame window following the snapshot
        sp_inner = SP(univ, start=num, stop=num+800, select=inner_layer_list)
        sp_mid   = SP(univ, start=num, stop=num+800, select=mid_layer_list)
        sp_outer = SP(univ, start=num, stop=num+800, select=outer_layer_list)

        all_inner.append(sp_inner)
        all_mid.append(sp_mid)
        all_outer.append(sp_outer)

    # Average across all windows element-wise (assumes SP returns array-like)
    avg_inner = np.mean(all_inner, axis=0)
    avg_mid   = np.mean(all_mid,   axis=0)
    avg_outer = np.mean(all_outer, axis=0)

    # Save averaged results
    df = pd.DataFrame({
        'Inner': avg_inner,
        'Mid':   avg_mid,
        'Outer': avg_outer
    })
    df.to_csv(f'data/sp_3res_{sys}.csv', index=False)
    print(f"Data successfully saved for system: {sys}")

In [ ]:
# 3. Main Comparison Loop
systems = ['charged', 'equal', 'aromatic', 'nonpolar', 'polar']
results = []

layers = ['Inner','Mid','Outer']


for sys in systems:
    plt.figure(figsize=(8, 5))
    
    # Load data
    try:
        loaded_df = pd.read_csv(f'data/sp_3res_{sys}.csv')
    except FileNotFoundError:
        print(f"File for {sys} not found, skipping...")
        continue
        
    # Calculate Teff for each layer and plot
    teff_inner = fit_constrained(loaded_df['Inner'].values, 'Inner')
    teff_mid   = fit_constrained(loaded_df['Mid'].values, 'Mid')
    teff_outer = fit_constrained(loaded_df['Outer'].values, 'Outer')

    # Store results for the final summary table
    results.append({
        'System': sys,
        'Inner_Teff': teff_inner,
        'Mid_Teff': teff_mid,
        'Outer_Teff': teff_outer
    })

    plt.xlabel('Time')
    plt.ylabel('Normalized SP')
    plt.title(f'Constrained Double Exponential Fit - System: {sys.capitalize()}')
    plt.legend()
    plt.show()

# 4. Final Comparison Table
summary_df = pd.DataFrame(results)
print("\n--- Final Teff Comparison Summary ---")
print(summary_df.to_string(index=False))

# Optional: Plot the comparison of Teff across systems
summary_df.set_index('System').plot(kind='bar', figsize=(10, 6))
plt.ylabel('Effective Mean Residence Time (Teff)')
plt.title('Comparison of Teff across Chemical Systems and Layers')
plt.xticks(rotation=45)
plt.show()

In [ ]:

import seaborn as sns

# Assuming 'summary_df' was created in the previous step
# Melt the dataframe for easier plotting with Seaborn
melted_df = summary_df.melt(id_vars='System', var_name='Layer', value_name='Teff')

# Define the layers to plot separately
layers = ['Inner_Teff', 'Mid_Teff', 'Outer_Teff']

for layer_name in layers:
    plt.figure(figsize=(9, 5))
    
    # Filter data for the specific layer
    layer_data = summary_df[['System', layer_name]].sort_values(by=layer_name, ascending=False)
    
    # Create bar plot
    sns.barplot(x='System', y=layer_name, data=layer_data, palette='viridis')
    
    plt.title(f'Comparison of $T_{{eff}}$ across Systems: {layer_name.split("_")[0]} Layer')
    plt.ylabel('Mean Residence Time ($T_{eff}$)')
    plt.xlabel('Chemical System')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.show()

# Print the numeric breakdown
for layer in layers:
    print(f"\n--- Statistics for {layer} ---")
    print(summary_df[['System', layer]].sort_values(by=layer, ascending=False).to_string(index=False))

In [ ]:

# 3. Main Comparison Loop
systems = ['charged', 'polar', 'equal', 'aromatic', 'nonpolar']
#systems = ['2_close', '2_spread', '3_close', '3_spread']
results = []

layers = ['Inner','Mid','Outer']


for lay in layers:
    plt.figure(figsize=(4,3))
    for sys in systems:
        loaded_df = pd.read_csv(f'data/sp_9res_{sys}.csv')
        
        plt.plot(loaded_df[lay],label=sys)

    plt.legend()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

systems = ['charged', 'equal', 'aromatic', 'nonpolar', 'polar']
layers = ['Inner', 'Mid', 'Outer']

results = []

# ---- LOOP OVER SYSTEMS ----
for sys in systems:

    # Load both datasets
    try:
        df_9 = pd.read_csv(f'data/sp_9res_{sys}.csv')
        df_3 = pd.read_csv(f'data/sp_3res_{sys}.csv')
    except FileNotFoundError:
        print(f"Missing file for {sys}, skipping...")
        continue

    # ---- COMPUTE Teff ----
    teff_9 = {
        'Inner': fit_constrained(df_9['Inner'].values, 'Inner'),
        'Mid':   fit_constrained(df_9['Mid'].values, 'Mid'),
        'Outer': fit_constrained(df_9['Outer'].values, 'Outer')
    }

    teff_3 = {
        'Inner': fit_constrained(df_3['Inner'].values, 'Inner'),
        'Mid':   fit_constrained(df_3['Mid'].values, 'Mid'),
        'Outer': fit_constrained(df_3['Outer'].values, 'Outer')
    }

    # ---- STORE ----
    for layer in layers:
        results.append({
            'System': sys,
            'Layer': layer,
            'Teff_9res': teff_9[layer],
            'Teff_3res': teff_3[layer]
        })

# ---- CREATE DATAFRAME ----
summary_df = pd.DataFrame(results)

print("\n--- Combined Teff Comparison ---")
print(summary_df.to_string(index=False))

In [ ]:
for layer in layers:
    subset = summary_df[summary_df['Layer'] == layer]

    x = range(len(subset))

    plt.figure(figsize=(8,5))
    plt.bar(x, subset['Teff_9res'], width=0.4, label='9res', align='center')
    plt.bar([i + 0.4 for i in x], subset['Teff_3res'], width=0.4, label='3res')

    plt.xticks([i + 0.2 for i in x], subset['System'], rotation=45)
    plt.ylabel('Teff')
    plt.title(f'{layer} Layer: 3res vs 9res')
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
summary_df['Teff_ratio (9/3)'] = summary_df['Teff_9res'] / summary_df['Teff_3res']

for layer in layers:
    subset = summary_df[summary_df['Layer'] == layer]

    plt.figure(figsize=(8,5))
    plt.bar(subset['System'], subset['Teff_ratio (9/3)'])
    plt.axhline(1, linestyle='--')  # reference line

    plt.ylabel('Teff Ratio (9res / 3res)')
    plt.title(f'{layer} Layer: Residence Time Amplification')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
pivot = summary_df.pivot(index='System', columns='Layer', values='Teff_3res')
pivot.plot(kind='bar', figsize=(10,6))
plt.title('9res: Teff across layers')
plt.ylabel('Teff')
plt.xticks(rotation=45)
plt.show()

In [ ]:
pivot = summary_df.pivot(index='System', columns='Layer', values='Teff_9res')
pivot.plot(kind='bar', figsize=(10,6))
plt.title('9res: Teff across layers')
plt.ylabel('Teff')
plt.xticks(rotation=45)
plt.show()

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

systems = ['charged', 'equal', 'aromatic', 'nonpolar', 'polar']
layers = ['Inner', 'Mid', 'Outer']

for sys in systems:
    try:
        df_9 = pd.read_csv(f'data/sp_9res_{sys}.csv')
        df_3 = pd.read_csv(f'data/sp_3res_{sys}.csv')
    except FileNotFoundError:
        print(f"Missing {sys}")
        continue

    time = range(len(df_9))  # assuming same length

    for layer in layers:
        plt.figure(figsize=(7,5))

        plt.plot(time, df_9[layer], label='9res', linewidth=2)
        plt.plot(time, df_3[layer], '--', label='3res', linewidth=2)

        plt.xlabel('Time')
        plt.ylabel('Survival Probability')
        plt.title(f'{sys.capitalize()} | {layer} Layer')
        plt.legend()
        plt.tight_layout()
        plt.show()

In [ ]:
sys='charged'

stru = parmed.load_file(f'../gxg_comps/3_res/{sys}/output.top',
                        xyz=f'../gxg_comps/3_res/{sys}/output.gro')
univ = mda.Universe(stru, f'../gxg_comps/3_res/{sys}/trajectory.dcd')

In [ ]:
num=100000

inner_layers, outer_layers, res_list = surface_comp3(
    univ, start=num, stop=num+1, step=1
)

In [ ]:
select = ' '.join(str(item) for item in inner_layers)
select_2 = ' '.join(str(item) for item in outer_layers)


In [ ]:
start = 100000

dist_water_inner = 3.4
dist_water_outer = 3.0

# --- go to initial frame ---
univ.trajectory[start]


inner_water = univ.select_atoms(
    f"name OW and (around {dist_water_inner} (protein and resid {select}))"
)
outer_water = univ.select_atoms(
    f"name OW and (around {dist_water_outer} (protein and resid {select_2}))"
)

# --- remove waters that appear in both shells ---

inner_ids = set(inner_water.indices)
outer_ids = set(outer_water.indices)

common_ids = inner_ids.intersection(outer_ids)
inner_ids = inner_ids - common_ids

In [ ]:
len(inner_ids),len(common_ids),len(outer_ids)

In [ ]:
len(np.unique(univ.select_atoms('resname LYS').resids)
   )

In [ ]:
sys='charged'

stru = parmed.load_file(f'../gxg_comps/9_res/{sys}/output.top',
                        xyz=f'../gxg_comps/9_res/{sys}/output.gro')
univ = mda.Universe(stru, f'../gxg_comps/9_res/{sys}/traj.dcd')

In [ ]:
len(np.unique(univ.select_atoms('resname LYS').resids)
   )

In [ ]:
# 3. Main Comparison Loop
systems = ['charged', 'polar', 'equal', 'aromatic', 'nonpolar']

layers = ['Inner','Mid','Outer']


for sys in systems:

    stru = parmed.load_file(f'../gxg_comps/9_res/{sys}/output.top',
                           xyz=f'../gxg_comps/9_res/{sys}/output.gro')

    # or explicitly by atom names:
    #stru.strip('@OW,HW1,HW2')
    univ = mda.Universe(stru,
                     f'../gxg_comps/9_res/{sys}/traj.dcd',
                     #'../gxg_comps/3_res/charged//traj_2-nw.dcd'
                    )
    print(univ.trajectory)
    
    
    
    
    inner_layers,mid_layers,outer_layers,res_list = surface_comp2(univ, 
                                                              start=60000,
                                                              stop=60001, 
                                                              step=1)
    
        
    inner_layer_list = ' '
    for item in inner_layers:
        inner_layer_list += str(item) + ' '


    outer_layer_list = ' '
    for item in outer_layers:
        outer_layer_list += str(item) + ' '

    mid_layer_list = ' '
    for item in mid_layers:
        mid_layer_list += str(item) + ' '

    residue_list = ' '
    for item in res_list:
        residue_list += str(item) + ' '
        
   
    sp_inner_layer = SP(univ,start=60000,stop=60800,select=inner_layer_list)

    sp_mid_layer = SP(univ,start=60000,stop=60800,select=mid_layer_list)

    sp_outer_layer = SP(univ,start=60000,stop=60800,select=outer_layer_list)
    
    
    
    df = pd.DataFrame({
        'Inner': sp_inner_layer,
        'Mid': sp_mid_layer,
        'Outer': sp_outer_layer
    })

    # Save the DataFrame to a CSV file
    df.to_csv(f'data/sp_9res_{sys}.csv', index=False)
    print(f"Data successfully saved to {sys}")

    # ---------------    

